In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support import expected_conditions as EC
import pyautogui
import time
import pandas as pd
 
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


In [3]:
def dismiss_wuzzuf_popups(driver):
    time.sleep(1)

    width, height = pyautogui.size()
    pyautogui.click(width - 30, height - 80)

    time.sleep(1)

In [4]:
def searchRoutine(driver, key):
    driver.get("https://wuzzuf.net/jobs/egypt")
    time.sleep(5)
    driver.find_element(By.CSS_SELECTOR,".css-pp6392.e1n2h7jb1").send_keys(key)
    driver.switch_to.active_element.send_keys(Keys.ENTER)
    time.sleep(10)
    driver.refresh()
    time.sleep(10)
    dismiss_wuzzuf_popups(driver)
    time.sleep(10)
    dismiss_wuzzuf_popups(driver)

    Jobs = []

    page = True

    while page:
        jobsPostings = driver.find_elements(By.CLASS_NAME, "css-pkv5jc")
        for job in jobsPostings:
            try: 
                title = job.find_element(By.CLASS_NAME, "css-o171kl").text.strip()
                company = job.find_element(By.CLASS_NAME, "css-ipsyv7").text.strip()
                location = job.find_element(By.CLASS_NAME, "css-16x61xq").text.strip()
                type = job.find_element(By.CLASS_NAME, "css-5jhz9n").text.strip()
                exps_1 = job.find_elements(By.CLASS_NAME, "css-o171kl")
                exps_2 = job.find_elements(By.CLASS_NAME, "css-5x9pm1")
                experience = []
                for e in exps_1:
                    experience.append(e.text.strip())
                for e in exps_2:
                    experience.append(e.text.strip())
                experience = " ".join(experience)
                url = job.find_element(By.CLASS_NAME, "css-o171kl").get_attribute("href")

                Jobs.append({
                    "Title": title,
                    "Company": company,
                    "Location": location,
                    "Type": type.replace("\n", " | "),
                    "Experience": experience,
                    "URL": url
                })
            except Exception: 
                pass

        try:
            driver.find_element(By.CSS_SELECTOR,'a[aria-label="Next page"]').click()
        except:
            page = False
        time.sleep(5)
    return Jobs



In [5]:
driver = webdriver.Chrome(options=options)
keys = ["Data Engineer", "Data Science", "AI Engineer", "Machine Learning Engineer"]
Jobs = [] 

for key in keys: 
    Jobs += searchRoutine(driver,key)
Jobs


[{'Title': 'Mechanical Engineer - Data Center Cooling & Critical Infrastructure',
  'Company': 'UNI Power -',
  'Location': '6th of October, Giza, Egypt',
  'Type': 'Full Time | On-site',
  'Experience': 'Mechanical Engineer - Data Center Cooling & Critical Infrastructure Experienced · Sales/Retail · Engineering - Mechanical/Electrical Mechanical Engineering Engineering Design · CAD Software · Project Management · Technical Sales · Client Relationship Management · Problem Solving · Communication Skills',
  'URL': 'https://wuzzuf.net/jobs/p/gunnvcfdmj3a-mechanical-engineer-data-center-cooling-critical-infrastructure-uni-power-giza-egypt'},
 {'Title': 'Associate System Engineer “Associate Data Management Engineer',
  'Company': 'Target Integrated Systems -',
  'Location': 'Cairo, Egypt',
  'Type': 'Full Time | On-site',
  'Experience': 'Associate System Engineer “Associate Data Management Engineer Entry Level · Installation/Maintenance/Repair · IT/Software Development · Engineering - Tel

In [6]:
df = pd.DataFrame(Jobs)
df

,Title,Company,Location,Type,Experience,URL
0,Mechanical Engineer - Data Center Cooling & Cr...,UNI Power -,"6th of October, Giza, Egypt",Full Time | On-site,Mechanical Engineer - Data Center Cooling & Cr...,https://wuzzuf.net/jobs/p/gunnvcfdmj3a-mechani...
1,Associate System Engineer “Associate Data Mana...,Target Integrated Systems -,"Cairo, Egypt",Full Time | On-site,Associate System Engineer “Associate Data Mana...,https://wuzzuf.net/jobs/p/ubsrja3n94m1-associa...
2,Senior Data Engineer,Vertex Technologies -,"Cairo, Egypt",Full Time | On-site,Senior Data Engineer Experienced · IT/Software...,https://wuzzuf.net/jobs/p/3x6prdwpt2tz-senior-...
3,Data Center Facility Electrical/ Mechanical En...,GPX Egypt -,"New Cairo, Cairo, Egypt",Full Time | On-site,Data Center Facility Electrical/ Mechanical En...,https://wuzzuf.net/jobs/p/39xm37penfcr-data-ce...
4,Data Engineer (Infrastructure & Operations),MEAHCO - Saudi German Health -,"Katameya, Cairo, Egypt",Full Time | On-site,Data Engineer (Infrastructure & Operations) Ex...,https://wuzzuf.net/jobs/p/mhqqwhx31ryd-data-en...
...,...,...,...,...,...,...
878,GIS Technician,Citylogix ME -,"Katameya, Cairo, Egypt",Full Time | On-site,GIS Technician Entry Level · IT/Software Devel...,https://wuzzuf.net/jobs/p/hlkkamp5vekw-gis-tec...
879,Junior System Administrator,SHM Industries -,"Cairo, Egypt",Full Time | On-site,Junior System Administrator Entry Level · IT/S...,https://wuzzuf.net/jobs/p/esh3hhmwpxnt-junior-...
880,Maintenance & Technical Support Specialist,Infinity tecs -,"Cairo, Egypt",Internship | On-site,Maintenance & Technical Support Specialist Ent...,https://wuzzuf.net/internship/okjfuagtatr5-mai...
881,Python Developer (CAD Automation),StoryGold -,"San Francisco, United States",Full Time | Remote,Python Developer (CAD Automation) Entry Level ...,https://wuzzuf.net/jobs/p/hzdsg3stdbez-python-...


In [7]:
df['Company'] = df['Company'].str.removesuffix(" -")

In [8]:
df.to_csv('wuzzuf_jobs.csv')